In [ ]:
# ==========================================
# 14 - FEATURE ENGINEERING EXPERIMENT
# KHỞI TẠO MÔI TRƯỜNG
# ==========================================

import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

RANDOM_SEED = 42
lib.np.random.seed(RANDOM_SEED)

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

subjects = [
    "van", "toan", "ngoai_ngu_1", "gdcd",
    "lich_su_dia_li", "khtn", "cong_nghe", "tin_hoc"
]

grades = [6, 7, 8, 9]

data_path = ROOT_DIR / "data/data-splits/development.csv"

output_dir = (
    ROOT_DIR
    / "experiments/v2_optimization/03_feature_engineering/results"
)

table_dir = output_dir / "tables"
config_dir = output_dir / "configs"

for directory in [table_dir, config_dir]:
    directory.mkdir(parents=True, exist_ok=True)

# Chỉ đọc Development
df_dev = lib.pd.read_csv(
    data_path,
    encoding="utf-8-sig"
)

score_cols = [
    col for col in df_dev.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

X_original = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()
student_ids = df_dev[id_col].copy()

assert X_original.shape == (9000, 96)
assert y.shape == (9000, 9)
assert X_original.notna().all().all()

# Chứa các đặc trưng được tạo thêm
X_engineered = lib.pd.DataFrame(
    index=X_original.index
)

feature_groups = {}

def register_features(group_name, columns_before):
    feature_groups[group_name] = [
        c for c in X_engineered.columns
        if c not in columns_before
    ]

print("Development:", len(df_dev))
print("Original features:", X_original.shape[1])
print("Target labels:", y.shape[1])
print("PASS: Khởi tạo thành công.")

In [ ]:
# ==========================================
# FEATURE GROUP 1
# ĐIỂM TRUNG BÌNH 4 NĂM
# ==========================================

before = set(X_engineered.columns)

for subject in subjects:

    cols = [
        f"L{grade}_{subject}_ca_nam"
        for grade in grades
    ]

    X_engineered[f"FE_{subject}_mean_4years"] = (
        X_original[cols].mean(axis=1)
    )

# Điểm trung bình tất cả môn trong 4 năm
all_yearly_cols = [
    f"L{grade}_{subject}_ca_nam"
    for grade in grades
    for subject in subjects
]

X_engineered["FE_overall_mean_4years"] = (
    X_original[all_yearly_cols].mean(axis=1)
)

register_features("mean_4years", before)

print("Số đặc trưng mới:", len(feature_groups["mean_4years"]))
display(X_engineered[feature_groups["mean_4years"]].head())

In [ ]:
# ==========================================
# FEATURE GROUP 2
# ĐỘ LỆCH CHUẨN QUA 4 NĂM
# ==========================================

before = set(X_engineered.columns)

for subject in subjects:

    cols = [
        f"L{grade}_{subject}_ca_nam"
        for grade in grades
    ]

    X_engineered[f"FE_{subject}_std_4years"] = (
        X_original[cols].std(axis=1, ddof=0)
    )

# Độ lệch chuẩn của điểm trung bình từng năm
yearly_means = lib.pd.DataFrame({
    f"L{grade}": X_original[
        [
            f"L{grade}_{subject}_ca_nam"
            for subject in subjects
        ]
    ].mean(axis=1)
    for grade in grades
})

X_engineered["FE_overall_std_4years"] = (
    yearly_means.std(axis=1, ddof=0)
)

register_features("std_4years", before)

print("Số đặc trưng mới:", len(feature_groups["std_4years"]))
display(X_engineered[feature_groups["std_4years"]].head())

In [ ]:
# ==========================================
# FEATURE GROUP 3
# CHÊNH LỆCH ĐIỂM LỚP 6–9
# ==========================================

before = set(X_engineered.columns)

for subject in subjects:

    X_engineered[f"FE_{subject}_delta_L9_L6"] = (
        X_original[f"L9_{subject}_ca_nam"]
        - X_original[f"L6_{subject}_ca_nam"]
    )

X_engineered["FE_overall_delta_L9_L6"] = (
    yearly_means["L9"] - yearly_means["L6"]
)

register_features("delta_L9_L6", before)

print("Số đặc trưng mới:", len(feature_groups["delta_L9_L6"]))
display(X_engineered[feature_groups["delta_L9_L6"]].head())

In [ ]:
# ==========================================
# FEATURE GROUP 4
# MỨC TĂNG/GIẢM ĐIỂM LIÊN TIẾP
# ==========================================

before = set(X_engineered.columns)

grade_pairs = [(6, 7), (7, 8), (8, 9)]

for subject in subjects:

    for grade_a, grade_b in grade_pairs:

        X_engineered[
            f"FE_{subject}_change_L{grade_a}_L{grade_b}"
        ] = (
            X_original[f"L{grade_b}_{subject}_ca_nam"]
            - X_original[f"L{grade_a}_{subject}_ca_nam"]
        )

# Mức tăng giảm điểm trung bình chung
for grade_a, grade_b in grade_pairs:

    X_engineered[
        f"FE_overall_change_L{grade_a}_L{grade_b}"
    ] = (
        yearly_means[f"L{grade_b}"]
        - yearly_means[f"L{grade_a}"]
    )

register_features("yearly_changes", before)

print("Số đặc trưng mới:", len(feature_groups["yearly_changes"]))
display(X_engineered[feature_groups["yearly_changes"]].head())

In [ ]:
# ==========================================
# FEATURE GROUP 5
# XU HƯỚNG ĐIỂM QUA 4 NĂM
# ==========================================

before = set(X_engineered.columns)

# Trục thời gian cố định: lớp 6, 7, 8, 9
time_axis = lib.np.array(grades, dtype=float)

time_centered = time_axis - time_axis.mean()
denominator = lib.np.sum(time_centered ** 2)

def calculate_trend(values):

    # values: (n_students, 4)
    return (
        values @ time_centered
    ) / denominator


for subject in subjects:

    cols = [
        f"L{grade}_{subject}_ca_nam"
        for grade in grades
    ]

    values = X_original[cols].to_numpy(dtype=float)

    X_engineered[f"FE_{subject}_trend"] = (
        calculate_trend(values)
    )

X_engineered["FE_overall_trend"] = (
    calculate_trend(
        yearly_means.to_numpy(dtype=float)
    )
)

register_features("trend", before)

print("Số đặc trưng mới:", len(feature_groups["trend"]))
display(X_engineered[feature_groups["trend"]].head())

In [ ]:
# ==========================================
# FEATURE GROUP 6
# ĐIỂM TRUNG BÌNH THEO NHÓM MÔN
# ==========================================

before = set(X_engineered.columns)

subject_groups = {
    "natural": ["toan", "khtn"],
    "social": ["van", "lich_su_dia_li", "gdcd"],
    "technology": ["cong_nghe", "tin_hoc"],
    "foreign_language": ["ngoai_ngu_1"]
}

for group_name, group_subjects in subject_groups.items():

    # Điểm trung bình nhóm môn theo từng lớp
    for grade in grades:

        cols = [
            f"L{grade}_{subject}_ca_nam"
            for subject in group_subjects
        ]

        X_engineered[
            f"FE_{group_name}_mean_L{grade}"
        ] = X_original[cols].mean(axis=1)

    # Điểm trung bình nhóm môn trong 4 năm
    cols_4years = [
        f"L{grade}_{subject}_ca_nam"
        for grade in grades
        for subject in group_subjects
    ]

    X_engineered[
        f"FE_{group_name}_mean_4years"
    ] = X_original[cols_4years].mean(axis=1)

register_features("subject_groups", before)

print("Số đặc trưng mới:", len(feature_groups["subject_groups"]))
display(X_engineered[feature_groups["subject_groups"]].head())

In [ ]:
# ==========================================
# FEATURE GROUP 7
# MỨC ĐỘ ỔN ĐỊNH KẾT QUẢ HỌC TẬP
# ==========================================

before = set(X_engineered.columns)

for subject in subjects:

    cols = [
        f"L{grade}_{subject}_ca_nam"
        for grade in grades
    ]

    values = X_original[cols].to_numpy(dtype=float)

    # Biên độ dao động điểm
    X_engineered[
        f"FE_{subject}_range_4years"
    ] = (
        values.max(axis=1)
        - values.min(axis=1)
    )

    # Trung bình mức thay đổi tuyệt đối giữa các năm
    X_engineered[
        f"FE_{subject}_mean_abs_change"
    ] = lib.np.abs(
        lib.np.diff(values, axis=1)
    ).mean(axis=1)


overall_values = yearly_means.to_numpy(dtype=float)

X_engineered["FE_overall_range_4years"] = (
    overall_values.max(axis=1)
    - overall_values.min(axis=1)
)

X_engineered["FE_overall_mean_abs_change"] = (
    lib.np.abs(
        lib.np.diff(overall_values, axis=1)
    ).mean(axis=1)
)

register_features("stability", before)

print("Số đặc trưng mới:", len(feature_groups["stability"]))
display(X_engineered[feature_groups["stability"]].head())

In [ ]:
# ==========================================
# TỔNG HỢP VÀ KIỂM ĐỊNH FEATURE ENGINEERING
# ==========================================

X_fe = lib.pd.concat(
    [X_original, X_engineered],
    axis=1
)

feature_report = lib.pd.DataFrame([
    {
        "Nhóm đặc trưng": group,
        "Số đặc trưng": len(columns)
    }
    for group, columns in feature_groups.items()
])

total_new = X_engineered.shape[1]

print("=" * 60)
print("FEATURE ENGINEERING VALIDATION")
print("=" * 60)

display(feature_report)

print(f"Đặc trưng gốc: {X_original.shape[1]}")
print(f"Đặc trưng mới: {total_new}")
print(f"Tổng đặc trưng: {X_fe.shape[1]}")
print(f"Kích thước X_fe: {X_fe.shape}")

# Kiểm tra cấu trúc
assert X_fe.shape == (9000, 197)
assert X_fe.columns.is_unique
assert X_fe.notna().all().all()

assert all(
    lib.pd.api.types.is_numeric_dtype(dtype)
    for dtype in X_fe.dtypes
)

assert lib.np.isfinite(
    X_fe.to_numpy(dtype=float)
).all()

assert not (set(X_fe.columns) & set(choice_cols))
assert id_col not in X_fe.columns
assert X_fe.index.equals(y.index)

# Kiểm tra các đặc trưng gốc không bị thay đổi
lib.pd.testing.assert_frame_equal(
    X_fe[score_cols],
    X_original
)

print("\nPASS: Đặc trưng mới hợp lệ.")
print("PASS: Không sử dụng nhãn mục tiêu hoặc ID.")
print("PASS: Không thay đổi 96 đặc trưng gốc.")

In [ ]:
# ==========================================
# SAVE FEATURE ENGINEERING RESULTS
# ==========================================

# Lưu riêng X, y, ID
X_fe.to_csv(
    table_dir / "X_development_feature_engineered.csv",
    index=False,
    encoding="utf-8-sig"
)

student_ids.to_frame(name=id_col).to_csv(
    table_dir / "feature_engineering_student_ids.csv",
    index=False,
    encoding="utf-8-sig"
)

feature_report.to_csv(
    table_dir / "feature_groups_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# Danh sách đặc trưng
feature_list = lib.pd.DataFrame({
    "Feature": X_fe.columns,
    "Type": [
        "Original" if col in score_cols else "Engineered"
        for col in X_fe.columns
    ]
})

feature_list.to_csv(
    table_dir / "feature_list.csv",
    index=False,
    encoding="utf-8-sig"
)

# Lưu cấu hình
config = {
    "experiment": "14 - Feature Engineering",
    "random_seed": RANDOM_SEED,
    "data_source": "Development only",
    "n_students": len(X_fe),
    "original_features": len(score_cols),
    "engineered_features": X_engineered.shape[1],
    "total_features": X_fe.shape[1],
    "feature_groups": feature_groups,
    "subject_groups": subject_groups,
    "target_used_for_engineering": False,
    "independent_test_used": False
}

with open(
    config_dir / "feature_engineering_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

print("PASS: Đã lưu kết quả Feature Engineering.")
print("Thư mục:", output_dir.resolve())